In [ ]:
!pip install mcstastox

In [ ]:
%load_ext autoreload
%autoreload 2
import os
import sys
module_path = os.path.abspath(os.path.join('../.local/lib/python3.12/site-packages/')) # or the path to your source code
sys.path.insert(0, module_path)

In [ ]:
import mcstasscript as ms
from templateSANS_Mantid_generated import make

In [ ]:
instr=make()

In [ ]:
# Show diagram
instr.show_diagram()

In [ ]:
instr.settings(output_path = 'test', ncount = 1e8, mpi = 12, NeXus = True, checks = False)

In [ ]:
# Generate a dataset with default parameters.
data = instr.backengine()

## Step 2: Conversion to Scipp object using McStasToX

The data in NeXus format created in the virtual experiment need to be transformed in a scipp object that will be used in the data processing.

First, we open the file by specifying the folder where the *mccode.h5* file is located.

In [ ]:
file_path = "test"
%matplotlib widget

We import mcstastox and read all the components in the simulation:

In [ ]:
import mcstastox

with mcstastox.Read(file_path) as loaded_data:
    loaded_data.show_components()

Now we select the components that have events with IDs:

In [ ]:
with mcstastox.Read(file_path) as loaded_data:
    loaded_data.show_components_with_ids()

We get these components and we exported together with the source and sample components, becoming a scipp object that can be evaluated further.

In [ ]:
with mcstastox.Read(file_path) as loaded_data:
    print(loaded_data.get_components_with_ids())

In [ ]:
with mcstastox.Read(file_path) as loaded_data:
    loaded_data.check_id_continuous()

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def plot(array_3d, small_arrays=None, points=None):
    # Create the figure
    fig = plt.figure()
    ax = fig.add_subplot(111, projection='3d')

    # Plot the values
    every = 1
    ax.scatter(
        array_3d[::every, 2],
        array_3d[::every, 0],
        array_3d[::every, 1],
        c=range(0, len(array_3d[::every, 0])),
        marker='.',
    )

    if small_arrays is not None:
        for small_array in small_arrays:
            ax.scatter(
                small_array[:, 2],
                small_array[:, 0],
                small_array[:, 1],
                c='b',
                marker='.',
            )

    if points is not None:
        for point in points:
            ax.scatter(point[2], point[0], point[1], c='k', marker='.')

    ax.set_xlabel('Z-axis')
    ax.set_ylabel('X-axis')
    ax.set_zlabel('Y-axis')

    plt.show()

In [ ]:
with mcstastox.Read(file_path) as loaded_data:
    coordinates = loaded_data.get_id_to_global_coordinates()
    sample_pos = loaded_data.get_global_component_coordinates("sampleMantid")
    source_pos = loaded_data.get_global_component_coordinates("sourceMantid")

plot(coordinates)

In [ ]:
with mcstastox.Read(file_path) as loaded_data:
    scipp_object = loaded_data.export_scipp(source_name="sourceMantid",
                                            sample_name="sampleMantid")

In [ ]:
scipp_object

## Step 3: Visualization and data transformation

For visualization of the pixels we need to import plopp

In [ ]:
import plopp as pp

pp.scatter3d(scipp_object["events"].hist(), pos='position', size=0.01, cbar=True, norm="linear")

Scippneutron will allow us to do the transformation from time-of-flight to d-spacing

In [ ]:
from scippneutron.conversion.graph.beamline import beamline
from scippneutron.conversion.graph.tof import elastic

event_object = scipp_object["events"]

# McStas provides absolute time, not time of flight
event_object.bins.coords["tof"] = event_object.bins.coords["t"]

graph = {**beamline(scatter=True), **elastic("tof")}

In [ ]:
event_object = event_object.transform_coords("Q", graph=graph)

In [ ]:
event_object.hist(Q=1500).sum("pixel_id").plot(norm="linear")